# Advanced Landlab A: coupled landscape evolution

**Duration:** 2 hours, including a 10-minute break  
**Prerequisite:** complete `01_landlab_fundamentals.ipynb`  
**Goal:** couple flow routing, fluvial incision, uplift, and hillslope diffusion,
then test one parameter while holding the rest of the model constant.

By the end you should be able to explain not only *which* components run, but
why they run in a particular order and which fields connect them.

## 1. The scientific model and its component dependencies

We will represent two ways that topography changes:

- **Fluvial incision** lowers channels according to stream power,
  $I=K A^m S^n$.
- **Linear hillslope diffusion** moves material down local gradients,
  $q_s=-D
abla z$.

Uniform rock uplift raises core nodes between erosional updates.

| Component or operation | Reads | Creates or modifies |
| --- | --- | --- |
| uplift statement | core-node IDs, timestep | `topographic__elevation` |
| `FlowAccumulator` | `topographic__elevation`, boundaries | drainage area, receivers, upstream order, discharge |
| `FastscapeEroder` | elevation and flow-routing fields | `topographic__elevation` |
| `LinearDiffuser` | `topographic__elevation` | elevation and link gradient/flux fields |

The shared state is `topographic__elevation`. Flow-routing fields describe the
current surface and become stale whenever erosion or diffusion changes it.

> **Remember — component order follows data dependencies**
>
> Uplift changes elevation, so route flow next. Fastscape then uses those routing fields to incise channels. Diffusion uses the resulting elevation to update hillslopes. Recompute flow once more before making final drainage diagnostics.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from landlab import RasterModelGrid, imshow_grid
from landlab.components import (
    ChannelProfiler,
    FastscapeEroder,
    FlowAccumulator,
    LinearDiffuser,
)

### Flow routing and stream power

`FlowAccumulator(..., flow_director="D8")` sends flow from each raster node
toward its steepest downhill neighbor, considering four orthogonal and four
diagonal directions. D8 is therefore specific to raster geometry.

`FastscapeEroder` then uses drainage area $A$, steepest slope $S$, erodibility
$K$, and exponents $m,n$. The dimensions of $K$ depend on the chosen exponents,
so it should not be interpreted as a universal material constant.

> **Watch out — small noise is a modeling choice**
>
> A perfectly flat initial surface contains ties in flow direction. Small seeded random noise breaks those ties reproducibly, but it can influence where the first channels form. Hold the seed constant when comparing parameters.

## 2. Create a reproducible domain

In [ ]:
def make_landscape_grid(seed=0):
    # A modest grid keeps repeated workshop experiments quick.
    model_grid = RasterModelGrid((60, 80), xy_spacing=50.0)
    model_grid.set_closed_boundaries_at_grid_edges(
        right_is_closed=True,
        top_is_closed=True,
        left_is_closed=True,
        bottom_is_closed=False,
    )

    elevation = model_grid.add_zeros("topographic__elevation", at="node")
    rng = np.random.default_rng(seed)
    elevation[:] = (
        0.001 * model_grid.y_of_node
        + 0.01 * rng.random(model_grid.number_of_nodes)
    )
    return model_grid, elevation


preview_grid, preview_z = make_landscape_grid()
print(f"grid shape: {preview_grid.shape}")
print(f"node spacing: {preview_grid.dx:g} m")
print(f"core nodes: {preview_grid.number_of_core_nodes}")
imshow_grid(preview_grid, preview_z, cmap="terrain", colorbar_label="Elevation (m)")

The left, right, and top edges are closed; the bottom edge is fixed-value and
provides outlets. This boundary configuration is part of the scientific setup,
not merely a plotting preference. Only core nodes are uplifted.

## 3. Establish a fluvial-only baseline

In [ ]:
def run_fluvial_landscape(
    K_sp=1.0e-5,
    uplift_rate=0.001,
    total_time=500_000.0,
    time_step=1_000.0,
    seed=0,
):
    model_grid, elevation = make_landscape_grid(seed=seed)
    initial_elevation = elevation.copy()

    flow = FlowAccumulator(model_grid, flow_director="D8")
    eroder = FastscapeEroder(
        model_grid,
        K_sp=K_sp,
        m_sp=0.5,
        n_sp=1.0,
    )

    times = [0.0]
    relief = [np.ptp(elevation[model_grid.core_nodes])]
    number_of_steps = int(total_time / time_step)

    for step in range(1, number_of_steps + 1):
        elevation[model_grid.core_nodes] += uplift_rate * time_step
        flow.run_one_step()
        eroder.run_one_step(time_step)

        if step % 50 == 0:
            times.append(step * time_step)
            relief.append(np.ptp(elevation[model_grid.core_nodes]))

    # Refresh flow fields so diagnostics match the final topography.
    flow.run_one_step()
    return {
        "grid": model_grid,
        "initial_elevation": initial_elevation,
        "final_elevation": elevation,
        "times": np.asarray(times),
        "relief": np.asarray(relief),
    }


fluvial_result = run_fluvial_landscape()
print(f"final relief: {fluvial_result['relief'][-1]:.1f} m")

The function creates a fresh grid on every call. This prevents the coupled run
from accidentally starting with the already evolved fluvial topography. The
stored initial array is an explicit copy; the final array remains attached to
the returned grid.

In [ ]:
def plot_landscape_result(result, title):
    model_grid = result["grid"]
    initial = result["initial_elevation"]
    final = result["final_elevation"]
    log_area = np.log10(model_grid.at_node["drainage_area"] + 1.0)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    plt.sca(axes[0])
    imshow_grid(model_grid, final, cmap="terrain", colorbar_label="Elevation (m)")
    axes[0].set_title("Final topography")
    plt.sca(axes[1])
    imshow_grid(
        model_grid,
        final - initial,
        cmap="RdBu",
        colorbar_label="Elevation change (m)",
    )
    axes[1].set_title("Final minus initial")
    plt.sca(axes[2])
    imshow_grid(model_grid, log_area, cmap="Blues", colorbar_label="log10 area")
    axes[2].set_title("Drainage area")
    fig.suptitle(title)
    plt.tight_layout()


plot_landscape_result(fluvial_result, "Flow routing + fluvial incision")

### Inspect the routing fields

The flow accumulator creates several fields required by `FastscapeEroder`.
Drainage area is easiest to view on a logarithmic scale because it spans orders
of magnitude.

In [ ]:
baseline_grid = fluvial_result["grid"]
for name in [
    "drainage_area",
    "flow__receiver_node",
    "flow__upstream_node_order",
    "topographic__steepest_slope",
]:
    values = baseline_grid.at_node[name]
    print(f"{name:35s} location=node shape={values.shape}")

### Channel profiles

`ChannelProfiler` uses the final drainage-area field to identify channel nodes.
The threshold is a modeling and visualization decision: lowering it includes
smaller tributaries.

In [ ]:
profiler = ChannelProfiler(
    baseline_grid,
    number_of_watersheds=3,
    minimum_channel_threshold=5_000.0,
)
profiler.run_one_step()
profiler.plot_profiles()
profiler.plot_profiles_in_map_view()

> **Model check — diagnose before coupling**
>
> Confirm finite elevations, continuous drainage-area patterns, sensible channel profiles, and expected outlet locations. Coupling another process will make a pre-existing problem harder to identify.

## Debugging checkpoint and break

With a partner, explain what would be wrong with each loop:

1. `FastscapeEroder` runs before flow has ever been routed.
2. Flow is routed once before the loop and never refreshed.
3. Every comparison calls `make_landscape_grid()` with a different seed.
4. Uplift is applied to every node, including fixed-value boundaries.

Then take a 10-minute break.

## 4. Main challenge: add hillslope diffusion

Complete the scaffold below. Use `diffusivity=0.1` m²/yr and
`deposit=False` for this detachment-limited example. Both erosion components
modify the same elevation field.

Why `deposit=False`? `FastscapeEroder` removes channel material without routing
sediment. Allowing the diffuser to deposit into those channels can create a
process mismatch and discontinuous drainage networks. This is a modeling
assumption to record, not a universal setting for diffusion.

In [ ]:
def run_coupled_landscape(
    diffusivity=0.1,
    K_sp=1.0e-5,
    uplift_rate=0.001,
    total_time=500_000.0,
    time_step=1_000.0,
    seed=0,
):
    model_grid, elevation = make_landscape_grid(seed=seed)
    initial_elevation = elevation.copy()

    flow = FlowAccumulator(model_grid, flow_director="D8")
    eroder = FastscapeEroder(
        model_grid, K_sp=K_sp, m_sp=0.5, n_sp=1.0
    )

    # TODO 1: instantiate LinearDiffuser. For this detachment-limited example,
    # use deposit=False and the diffusivity argument supplied to this function.

    times = [0.0]
    relief = [np.ptp(elevation[model_grid.core_nodes])]
    number_of_steps = int(total_time / time_step)

    for step in range(1, number_of_steps + 1):
        elevation[model_grid.core_nodes] += uplift_rate * time_step

        # TODO 2: route flow on the current topography.
        # TODO 3: run fluvial incision for one time step.
        # TODO 4: run hillslope diffusion for one time step.

        if step % 50 == 0:
            times.append(step * time_step)
            relief.append(np.ptp(elevation[model_grid.core_nodes]))

    # TODO 5: refresh flow fields after the final topographic change.

    return {
        "grid": model_grid,
        "initial_elevation": initial_elevation,
        "final_elevation": elevation,
        "times": np.asarray(times),
        "relief": np.asarray(relief),
    }


# After completing the TODOs, uncomment these lines:
# coupled_result = run_coupled_landscape()
# plot_landscape_result(coupled_result, "Coupled channel–hillslope model")

<details>
<summary><strong>Hint 1 — instantiate the component</strong></summary>

Pass the grid, `linear_diffusivity=diffusivity`, and `deposit=False` to
`LinearDiffuser`.
</details>

<details>
<summary><strong>Hint 2 — order inside the loop</strong></summary>

After uplift, call flow routing, fluvial incision with the timestep, and then
diffusion with the timestep.
</details>

<details>
<summary><strong>Hint 3 — final diagnostics</strong></summary>

Diffusion changes elevation after the last routing call. Run the flow
accumulator once more before plotting drainage area or creating a profiler.
</details>

> **Watch out — timestep and process order are coupled choices**
>
> A timestep acceptable for implicit stream-power incision may be too large for diffusion. If elevation oscillates, becomes non-finite, or drainage networks fragment, rerun from a fresh grid with a smaller timestep.

## 5. One-factor experiment

Before running, predict which diffusivity will produce lower relief. Compare
two runs that share the same seed, grid, uplift, erodibility, timestep, and
duration. Record a claim, evidence, and interpretation.

In [ ]:
# Choose two diffusivities. Keep every other argument unchanged.
low_diffusivity = 0.02
high_diffusivity = 0.5

# TODO: run both coupled experiments.
# low_D_result = run_coupled_landscape(diffusivity=low_diffusivity)
# high_D_result = run_coupled_landscape(diffusivity=high_diffusivity)

# TODO: plot both relief histories on one set of axes and interpret them.

**Prediction:**  
**Evidence:**  
**Interpretation:**  

Do not infer that the run with lower relief is “more correct.” This experiment
shows model sensitivity; validation would require observations and a comparison
metric.

## 6. Project handoff

Open `03_project_launch.ipynb`. A tractable project changes one dimension of
this working model—for example diffusivity, erodibility, uplift, or boundary
configuration—and chooses a diagnostic before running.

Record the random seed, grid and boundaries, component order, parameter values,
timestep, duration, and Landlab version. Restart the kernel and run the final
model from top to bottom before sharing it.

### Optional extensions

- Compare drainage density, channel steepness, or chi.
- Test a second defensible outlet configuration.
- Add a fault only after the baseline coupled model is reproducible.